In [1]:
import os
import json
import random
import shutil
from tqdm import tqdm

import numpy as np
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

DATA_ROOT = "/kaggle/input/datasets/sonlest/bom-dataset-v4/BOM-Dataset-v4"
ANNOT_PATH = f"{DATA_ROOT}/annotations/annotations.json"
IMAGE_DIR = f"{DATA_ROOT}/images"

WORK_DIR = "/kaggle/working/mmdet_dataset"
os.makedirs(WORK_DIR, exist_ok=True)

CLASSES = ["PartDrawing", "Note", "Table"]

In [2]:
def load_coco_json(path):
    with open(path, 'r') as f:
        data = json.load(f)
    return data

def split_dataset(data):
    images = data["images"]
    annotations = data["annotations"]

    image_ids = [img["id"] for img in images]

    train_ids, temp_ids = train_test_split(image_ids, test_size=0.4, random_state=42)
    val_ids, test_ids = train_test_split(temp_ids, test_size=0.25, random_state=42)

    return train_ids, val_ids, test_ids

In [3]:
def create_subset(data, image_ids):
    images = [img for img in data["images"] if img["id"] in image_ids]
    annotations = [ann for ann in data["annotations"] if ann["image_id"] in image_ids]

    return {
        "images": images,
        "annotations": annotations,
        "categories": data["categories"]
    }

In [4]:
def save_split_dataset(data, split_name):
    save_path = os.path.join(WORK_DIR, split_name)
    os.makedirs(save_path, exist_ok=True)

    # Save JSON
    json_path = os.path.join(save_path, f"{split_name}.json")
    with open(json_path, "w") as f:
        json.dump(data, f)

    # Copy images
    for img in tqdm(data["images"]):
        src = os.path.join(IMAGE_DIR, img["file_name"])
        dst = os.path.join(save_path, img["file_name"])
        shutil.copy(src, dst)

    return json_path, save_path

In [5]:
def install_mmdetection():
    !pip install -q openmim
    !mim install mmengine
    !mim install "mmcv>=2.0.0"
    !mim install mmdet

In [6]:
def train_model(train_json, val_json, train_dir, val_dir):
    from mmdet.apis import set_random_seed
    from mmengine.config import Config
    from mmdet.registry import RUNNERS

    cfg = Config.fromfile(
        '/usr/local/lib/python3.10/dist-packages/mmdet/configs/faster_rcnn/faster-rcnn_r50_fpn_1x_coco.py'
    )

    cfg.dataset_type = 'CocoDataset'

    cfg.data_root = WORK_DIR

    cfg.train_dataloader.dataset.ann_file = train_json
    cfg.train_dataloader.dataset.data_prefix = dict(img=train_dir)

    cfg.val_dataloader.dataset.ann_file = val_json
    cfg.val_dataloader.dataset.data_prefix = dict(img=val_dir)

    cfg.model.roi_head.bbox_head.num_classes = 3

    cfg.work_dir = "/kaggle/working/mmdet_output"
    cfg.train_cfg.max_epochs = 10
    cfg.train_dataloader.batch_size = 2

    cfg.val_evaluator.ann_file = val_json

    set_random_seed(42, deterministic=False)

    runner = RUNNERS.build(cfg)
    runner.train()

    return cfg

In [7]:
def run_inference(cfg, test_dir, test_json):
    from mmdet.apis import init_detector, inference_detector

    model = init_detector(cfg, cfg.work_dir + "/epoch_10.pth", device="cuda")

    with open(test_json, "r") as f:
        data = json.load(f)

    results = []

    for img_info in tqdm(data["images"]):
        img_path = os.path.join(test_dir, img_info["file_name"])
        image = cv2.imread(img_path)

        pred = inference_detector(model, img_path)

        objects = []

        for cls_id, bboxes in enumerate(pred.pred_instances):
            for bbox, score in zip(bboxes.bboxes, bboxes.scores):
                if score < 0.5:
                    continue

                x1, y1, x2, y2 = map(int, bbox)

                crop = image[y1:y2, x1:x2]

                # OCR placeholder
                ocr_text = "TODO_OCR"

                objects.append({
                    "class": CLASSES[cls_id],
                    "confidence": float(score),
                    "bbox": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
                    "ocr_content": ocr_text
                })

        results.append({
            "image": img_info["file_name"],
            "objects": objects
        })

    # Save JSON
    with open("/kaggle/working/final_output.json", "w") as f:
        json.dump(results, f, indent=2)

    print("✅ JSON OUTPUT SAVED")

In [8]:
def main():
    print("🚀 START PIPELINE")

    data = load_coco_json(ANNOT_PATH)

    train_ids, val_ids, test_ids = split_dataset(data)

    train_data = create_subset(data, train_ids)
    val_data = create_subset(data, val_ids)
    test_data = create_subset(data, test_ids)

    train_json, train_dir = save_split_dataset(train_data, "train")
    val_json, val_dir = save_split_dataset(val_data, "val")
    test_json, test_dir = save_split_dataset(test_data, "test")

    print("📦 Installing MMDetection...")
    install_mmdetection()

    print("🏋️ Training model...")
    cfg = train_model(train_json, val_json, train_dir, val_dir)

    print("🔍 Running inference...")
    run_inference(cfg, test_dir, test_json)

    print("🎯 DONE")


main()

🚀 START PIPELINE


100%|██████████| 18/18 [00:00<00:00, 143.05it/s]


📦 Installing MMDetection...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.4/239.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.5/506.5 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1

ModuleNotFoundError: No module named 'mmdet'